# 🚀 Lab 18: Clean and Analyze EMR Patient Data

## 📘 Lab Overview
In this lab, you will work with simulated hospital **Electronic Medical Record (EMR)** data and perform realistic healthcare data cleaning and analysis tasks. In real hospitals, patient records often come from multiple departments or different hospital systems, which means the data can have inconsistent column names, duplicate patient entries, missing values, and mixed formatting.

You will learn how to create multiple hospital datasets, clean and standardize them, merge them into one unified DataFrame, remove duplicate records, filter patient data by diagnosis, and compute outcome-based statistics such as average length of stay. These are practical data preparation and analytics skills used in healthcare informatics and medical data science.

This lab has been adapted for Google Colab so that everything runs directly inside a notebook environment without needing Linux terminal access, local files, or manual dataset uploads.

## 🎯 Learning Objectives
By the end of this lab, students will be able to:
* Clean and merge multiple hospital Electronic Medical Record (EMR) CSV files into a unified dataset.
* Remove duplicate records to ensure data integrity and accuracy.
* Fix column names to follow proper naming conventions for data analysis.
* Filter patient data based on specific medical diagnosis criteria.
* Group data by patient outcomes and compute statistical measures like average length of stay.
* Apply data preprocessing techniques commonly used in healthcare analytics.
* Use Python pandas library for efficient data manipulation and analysis.

## 🧰 Prerequisites
* Basic understanding of CSV file format and tabular data structure.
* Familiarity with Python programming fundamentals (variables, loops, functions).
* Basic knowledge of pandas library for data manipulation.
* Understanding of healthcare terminology (diagnosis, patient outcomes, length of stay).

## ⚙️ Environment Setup
This lab is designed to run fully inside Google Colab. We will ensure the necessary libraries are installed and imported.

In [1]:
# Step 1: Install necessary libraries (if not already present)
# We use %pip for Colab compatibility
%pip install pandas numpy

import os
import pandas as pd
import numpy as np

# Step 2: Verify environment
print("Python and libraries are ready.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# Step 3: Create a data directory to simulate a real-world file system
# This helps keep our workspace organized
os.makedirs("data", exist_ok=True)
print("Created data directory:", os.path.abspath("data"))

Python and libraries are ready.
Pandas version: 2.2.2
NumPy version: 2.0.2
Created data directory: /content/data


## 🏥 Creating EMR Data Files

### 💡 ELI10: What are we doing?
In the real world, different hospitals use different computer systems to store patient info. This means one hospital might call a column "Patient ID" while another calls it "ID". To practice fixing this, we are going to create three different "messy" files ourselves so we can learn how to fix them later!

We will create:
1. **Hospital A**: Standard names.
2. **Hospital B**: Lowercase names with underscores.
3. **Hospital C**: Messy names with extra spaces and symbols, plus some bad data (like negative stay days) to test our cleaning skills.

In [2]:
# --- Create Simulated EMR Data ---

# Hospital A dataset (Standard format)
hospital_a = pd.DataFrame([
    {'Patient ID': 'P001', 'Patient Name': 'John Smith', 'Diagnosis': 'Diabetes', 'Outcome': 'Recovered', 'Length of Stay': 5, 'Age': 45},
    {'Patient ID': 'P002', 'Patient Name': 'Mary Johnson', 'Diagnosis': 'Hypertension', 'Outcome': 'Improved', 'Length of Stay': 3, 'Age': 52},
    {'Patient ID': 'P003', 'Patient Name': 'Robert Brown', 'Diagnosis': 'Pneumonia', 'Outcome': 'Recovered', 'Length of Stay': 7, 'Age': 61},
    {'Patient ID': 'P004', 'Patient Name': 'Lisa White', 'Diagnosis': 'Fracture', 'Outcome': 'Transferred', 'Length of Stay': 4, 'Age': 30},
    {'Patient ID': 'P005', 'Patient Name': 'David Lee', 'Diagnosis': 'Heart Disease', 'Outcome': 'Critical', 'Length of Stay': 12, 'Age': 68},
    {'Patient ID': 'P006', 'Patient Name': 'Sarah Wilson', 'Diagnosis': 'Diabetes', 'Outcome': 'Improved', 'Length of Stay': 6, 'Age': 57},
    {'Patient ID': 'P007', 'Patient Name': 'Michael Davis', 'Diagnosis': 'Pneumonia', 'Outcome': 'Recovered', 'Length of Stay': 8, 'Age': 49},
    {'Patient ID': 'P008', 'Patient Name': 'Emma Brown', 'Diagnosis': 'Hypertension', 'Outcome': 'Recovered', 'Length of Stay': 2, 'Age': 44}
])

# Hospital B dataset (Different column names & shared Patient P002)
hospital_b = pd.DataFrame([
    {'patient_id': 'P009', 'full_name': 'James Taylor', 'diagnosis': 'Diabetes', 'outcome_status': 'Recovered', 'length_of_stay': 9, 'age': 63},
    {'patient_id': 'P010', 'full_name': 'Olivia Martinez', 'diagnosis': 'Heart Disease', 'outcome_status': 'Improved', 'length_of_stay': 10, 'age': 59},
    {'patient_id': 'P011', 'full_name': 'William Anderson', 'diagnosis': 'Pneumonia', 'outcome_status': 'Recovered', 'length_of_stay': 6, 'age': 40},
    {'patient_id': 'P002', 'full_name': 'Mary Johnson', 'diagnosis': 'Hypertension', 'outcome_status': 'Improved', 'length_of_stay': 3, 'age': 52}, # Duplicate
    {'patient_id': 'P012', 'full_name': 'Sophia Thomas', 'diagnosis': 'Asthma', 'outcome_status': 'Recovered', 'length_of_stay': 2, 'age': 26},
    {'patient_id': 'P013', 'full_name': 'Daniel Harris', 'diagnosis': 'Diabetes', 'outcome_status': 'Critical', 'length_of_stay': 14, 'age': 71},
    {'patient_id': 'P014', 'full_name': 'Ava Clark', 'diagnosis': 'Pneumonia', 'outcome_status': 'Transferred', 'length_of_stay': 5, 'age': 33},
    {'patient_id': 'P015', 'full_name': 'Benjamin Lewis', 'diagnosis': 'Hypertension', 'outcome_status': 'Recovered', 'length_of_stay': 4, 'age': 47}
])

# Hospital C dataset (Messy names, missing data, negative stay for P022, shared Patient P005)
hospital_c = pd.DataFrame([
    {'Patient-ID': 'P016', 'Name': 'Mia Walker', 'Primary Diagnosis': 'Heart Disease', 'Outcome ': 'Recovered', 'Length_of_Stay': '11', 'Age Years': 66},
    {'Patient-ID': 'P017', 'Name': 'Noah Hall', 'Primary Diagnosis': 'Diabetes', 'Outcome ': 'Improved', 'Length_of_Stay': '5', 'Age Years': 54},
    {'Patient-ID': 'P018', 'Name': 'Charlotte Allen', 'Primary Diagnosis': 'Pneumonia', 'Outcome ': 'Recovered', 'Length_of_Stay': '7', 'Age Years': 38},
    {'Patient-ID': 'P019', 'Name': 'Elijah Young', 'Primary Diagnosis': 'Hypertension', 'Outcome ': 'Recovered', 'Length_of_Stay': '3', 'Age Years': 50},
    {'Patient-ID': 'P020', 'Name': 'Amelia King', 'Primary Diagnosis': 'Diabetes', 'Outcome ': 'Critical', 'Length_of_Stay': '16', 'Age Years': 73},
    {'Patient-ID': 'P005', 'Name': 'David Lee', 'Primary Diagnosis': 'Heart Disease', 'Outcome ': 'Critical', 'Length_of_Stay': '12', 'Age Years': 68}, # Duplicate
    {'Patient-ID': 'P021', 'Name': 'Lucas Scott', 'Primary Diagnosis': 'Pneumonia', 'Outcome ': np.nan, 'Length_of_Stay': '9', 'Age Years': 58}, # Missing Outcome
    {'Patient-ID': 'P022', 'Name': 'Harper Green', 'Primary Diagnosis': 'Hypertension', 'Outcome ': 'Improved', 'Length_of_Stay': '-1', 'Age Years': 46} # Invalid Stay
])

# Save datasets as CSV files to the data/ folder
hospital_a.to_csv("data/hospital_a_patients.csv", index=False)
hospital_b.to_csv("data/hospital_b_patients.csv", index=False)
hospital_c.to_csv("data/hospital_c_patients.csv", index=False)

print("Sample EMR CSV files created successfully!")
print("Files in data folder:", os.listdir("data"))

Sample EMR CSV files created successfully!
Files in data folder: ['hospital_a_patients.csv', 'hospital_c_patients.csv', 'hospital_b_patients.csv']


## 🧹 Cleaning and Merging Hospital Data

### 💡 ELI10: What are we doing?
Now we have three files, but we can't analyze them together because the computer thinks "Full Name" and "Patient Name" are totally different things. We are going to write a script that "translates" all column names into one language so we can stack all the records into one big table.

In [3]:
# --- Subtask 2.1 & 2.2: Load and Examine ---
hospital_files = [
    'data/hospital_a_patients.csv',
    'data/hospital_b_patients.csv',
    'data/hospital_c_patients.csv'
]

hospital_datasets = []
for file in hospital_files:
    df = pd.read_csv(file)
    print(f"Loaded {file}: {df.shape[0]} rows")
    hospital_datasets.append(df)

# --- Subtask 2.3: Standardize Column Names ---
def clean_column_names(df):
    """Converts column names to lowercase and removes special characters."""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)
    return df

def harmonize_columns(df):
    """Maps varied column names to a single standardized EMR schema."""
    df = df.copy()
    rename_map = {
        'patientid': 'patient_id', 'patient_id': 'patient_id', 'patientid': 'patient_id',
        'patient_name': 'patient_name', 'full_name': 'patient_name', 'name': 'patient_name',
        'diagnosis': 'diagnosis', 'primary_diagnosis': 'diagnosis',
        'outcome': 'outcome', 'outcome_': 'outcome', 'outcome_status': 'outcome',
        'length_of_stay': 'length_of_stay', 'age': 'age', 'age_years': 'age'
    }
    return df.rename(columns=rename_map)

cleaned_datasets = []
for i, df in enumerate(hospital_datasets):
    cleaned_df = clean_column_names(df)
    cleaned_df = harmonize_columns(cleaned_df)
    cleaned_df['hospital_id'] = chr(65+i) # Mark which hospital the data came from
    cleaned_datasets.append(cleaned_df)

# --- Subtask 2.4: Merge ---
merged_emr_data = pd.concat(cleaned_datasets, ignore_index=True, sort=False)

# Ensure columns are in a specific order for cleanliness
expected_cols = ['patient_id', 'patient_name', 'diagnosis', 'outcome', 'length_of_stay', 'age', 'hospital_id']
merged_emr_data = merged_emr_data[expected_cols]

print("\nMerge Complete!")
display(merged_emr_data.head())

Loaded data/hospital_a_patients.csv: 8 rows
Loaded data/hospital_b_patients.csv: 8 rows
Loaded data/hospital_c_patients.csv: 8 rows

Merge Complete!


,patient_id,patient_name,diagnosis,outcome,length_of_stay,age,hospital_id
0,P001,John Smith,Diabetes,Recovered,5,45,A
1,P002,Mary Johnson,Hypertension,Improved,3,52,A
2,P003,Robert Brown,Pneumonia,Recovered,7,61,A
3,P004,Lisa White,Fracture,Transferred,4,30,A
4,P005,David Lee,Heart Disease,Critical,12,68,A


## 🔁 Removing Duplicate Patients

### 💡 ELI10: What are we doing?
Sometimes a patient goes to Hospital A AND Hospital B. If we just add the files together, that person appears twice! This makes our statistics wrong. We will find these double-entries using their unique `patient_id` and remove the extra copies.

In [4]:
# --- Subtask 3.1: Identify Duplicates ---
print(f"Total records before cleaning: {len(merged_emr_data)}")
unique_ids = merged_emr_data['patient_id'].nunique()
duplicate_count = len(merged_emr_data) - unique_ids
print(f"Found {duplicate_count} duplicate patient IDs.")

# Show the duplicates for verification
duplicates = merged_emr_data[merged_emr_data.duplicated(subset=['patient_id'], keep=False)]
print("\nDuplicate Records Found:")
display(duplicates.sort_values('patient_id'))

# --- Subtask 3.2: Remove Duplicates ---
# We keep 'first' to ensure we have one valid entry per patient
clean_emr_data = merged_emr_data.drop_duplicates(subset=['patient_id'], keep='first').copy()

print(f"\nTotal records after cleaning: {len(clean_emr_data)}")

Total records before cleaning: 24
Found 2 duplicate patient IDs.

Duplicate Records Found:


,patient_id,patient_name,diagnosis,outcome,length_of_stay,age,hospital_id
1,P002,Mary Johnson,Hypertension,Improved,3,52,A
11,P002,Mary Johnson,Hypertension,Improved,3,52,B
4,P005,David Lee,Heart Disease,Critical,12,68,A
21,P005,David Lee,Heart Disease,Critical,12,68,C



Total records after cleaning: 22


## 🔎 Filtering by Diagnosis

### 💡 ELI10: What are we doing?
Our hospital has lots of different types of patients (Broken bones, Asthma, etc.). Today, we only care about common chronic diseases like Diabetes or Heart Disease. We will "filter" the list to hide everyone else so we can focus our study.

In [5]:
# --- Subtask 4.1 & 4.2: Filter by Target Diagnoses ---
# Clean whitespace from text columns just in case
clean_emr_data['diagnosis'] = clean_emr_data['diagnosis'].astype(str).str.strip()

target_diagnoses = ['Diabetes', 'Hypertension', 'Heart Disease', 'Pneumonia']
pattern = '|'.join(target_diagnoses)

# Filter the data using a case-insensitive match
filtered_emr_data = clean_emr_data[clean_emr_data['diagnosis'].str.contains(pattern, case=False, na=False)].copy()

print(f"Filtered dataset contains {len(filtered_emr_data)} patients with target diagnoses.")
print("\nDiagnosis Breakdown:")
print(filtered_emr_data['diagnosis'].value_counts())

Filtered dataset contains 20 patients with target diagnoses.

Diagnosis Breakdown:
diagnosis
Diabetes         6
Pneumonia        6
Hypertension     5
Heart Disease    3
Name: count, dtype: int64


## 📊 Outcome-Based Analysis

### 💡 ELI10: What are we doing?
Finally, we want to know: "How long do Recovered patients stay versus Critical patients?"
To do this, we must:
1. Fix the `length_of_stay` numbers (change them from text to real math numbers).
2. Throw away impossible data (like staying for -1 days!).
3. Calculate the average stay for each outcome group.

In [6]:
# --- Subtask 5.1 & 5.2: Clean Outcome and Stay Data ---

# 1. Handle missing outcomes
filtered_emr_data = filtered_emr_data.dropna(subset=['outcome']).copy()

# 2. Convert Stay to numeric and remove invalid (negative) values
filtered_emr_data['length_of_stay'] = pd.to_numeric(filtered_emr_data['length_of_stay'], errors='coerce')

valid_los_data = filtered_emr_data[
    (filtered_emr_data['length_of_stay'].notnull()) &
    (filtered_emr_data['length_of_stay'] >= 0)
].copy()

# --- Subtask 5.3: Group and Compute Statistics ---
outcome_analysis = valid_los_data.groupby('outcome')['length_of_stay'].agg([
    'count', 'mean', 'median', 'min', 'max'
]).round(2)

print("Length of Stay Analysis by Patient Outcome:")
print("=" * 50)
display(outcome_analysis)

# Detailed view: Diagnosis AND Outcome
detailed_analysis = valid_los_data.groupby(['diagnosis', 'outcome'])['length_of_stay'].agg([
    'count', 'mean'
]).round(2)

# --- Subtask 5.4: Export Results ---
outcome_analysis.to_csv('emr_outcome_analysis.csv')
detailed_analysis.to_csv('emr_detailed_analysis.csv')
print("\nResults exported to CSV files successfully.")

Length of Stay Analysis by Patient Outcome:


,count,mean,median,min,max
outcome,,,,,
Critical,3,14.0,14.0,12,16
Improved,4,6.0,5.5,3,10
Recovered,10,6.2,6.5,2,11
Transferred,1,5.0,5.0,5,5



Results exported to CSV files successfully.


## ✅ Verification
Use this checklist to ensure your lab is correct:
- [x] Data files created in `data/` folder.
- [x] Column names harmonized (e.g., `Patient-ID` and `patient_id` merged).
- [x] Duplicate patient IDs removed (Check for P002 and P005).
- [x] Invalid data cleaned (Negative stay removed).
- [x] CSV reports generated.

## 🛠 Troubleshooting
* **Column Mismatch**: Ensure your `rename_map` covers all variations found in the raw data.
* **Numeric Errors**: Always use `pd.to_numeric(..., errors='coerce')` to prevent the code from crashing on text values in number columns.
* **Duplicates**: If duplicates remain, check if `subset=['patient_id']` was used correctly in `drop_duplicates()`.

## 📚 Key Takeaways
1. **Data cleaning is 80% of the job**: Real healthcare data is never perfect.
2. **Standardization is key**: You can't compare data until it speaks the same "language" (schema).
3. **Domain Knowledge Matters**: Knowing that a "negative stay" is impossible allows you to clean data effectively.

## 🏁 Conclusion
You have successfully built a pipeline that takes messy, multi-source hospital data and turns it into clean, actionable medical insights. This is the foundation of Healthcare Informatics!